# Portfolio-opdracht 2: Voorspel verhuuraantallen (CityBike)

**Deel: Opdracht 2 — Time series feature engineering** · Ahmed Bashabeeb

Dit notebook is het tweede deel van ons portfolio. Aan het eind worden de delen samengevoegd in
`Portfolio2_CityBike.ipynb`. De data (`train.csv`, `test.csv`) staat in de map `data/`. Het
inladen en opschonen van de data gebeurt met de code uit Opdracht 1 (`opdracht1_eda.ipynb`).

**Inhoud**

- Voorbereiding: imports, instellingen en data uit Opdracht 1
- 2.1 Evaluatiemethode · 2.2 Stationariteit (Dickey-Fuller) · 2.3 Trend
- 2.4 Seizoenspatronen met Fourier-analyse · 2.5 Cycli, autocorrelatie en lags
- 2.6 Overige tijdgebaseerde features · 2.7 Klasse `FeatureEngineer`
- 2.8 Invloed van de features · 2.9 Gekozen features

## Voorbereiding: imports, instellingen en data uit Opdracht 1

We gebruiken de klasse `DataLoader` (paragraaf 1.2) en de functie `add_calendar_features`
(paragraaf 1.5) uit Opdracht 1, en schonen de data op zoals in paragraaf 1.9: `hum == 0`
vervangen door interpolatie en 29–30 oktober 2012 verwijderen.

In [ ]:
# Bron: Week 10 – Regression Time Series 2 (import-cel)
# Bron: Week 11 – Time series analysis 3 (imports statsmodels)

# Importeren van de packages die nodig zijn voor de feature engineering
import os
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.fft import fft
from scipy.signal import find_peaks
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.deterministic import (CalendarFourier,
                                           DeterministicProcess)
from statsmodels.tsa.stattools import adfuller
from xgboost import XGBRegressor

# Waarschuwingen onderdrukken voor een overzichtelijk notebook
warnings.filterwarnings('ignore')

# Vaste instellingen
DATA_FOLDER = os.path.join('..', 'data')     # map met de Kaggle-bestanden
RANDOM_STATE = 42                            # reproduceerbare resultaten
HORIZON = 19 * 24                            # 456 uur, zoals de Kaggle-test
HOLDOUT_START = pd.Timestamp('2012-11-12')   # laatste 19 dagen = testset
pd.set_option('display.max_colwidth', 150)
plt.rcParams['figure.figsize'] = (12, 4)
sns.set_style('whitegrid')

In [ ]:
# Bron: Opdracht 1, paragraaf 1.2 (klasse DataLoader), 1.5
#       (add_calendar_features) en 1.9 (opschonen); opdracht1_eda.ipynb

class DataLoader:
    """Leest de bestanden van de Kaggle-competitie in en schoont ze op."""

    def __init__(self, folder='.'):
        self.folder = folder

    def load(self, file_name):
        """Lees een CSV-bestand in met 'date_hour' als datetime."""
        path = os.path.join(self.folder, file_name)
        return pd.read_csv(path, parse_dates=['date_hour'])

    @staticmethod
    def clean(data, outlier_days):
        """Schoon de data op (onderbouwing: paragraaf 1.4 en 1.9).

        - sorteren op tijd en dubbele uren verwijderen;
        - luchtvochtigheid 0 (meetfout) vervangen door interpolatie;
        - uitzonderlijke dagen (outliers) verwijderen.
        """
        data = data.sort_values('date_hour')
        data = data.drop_duplicates('date_hour').copy()
        data.loc[data['hum'] == 0, 'hum'] = np.nan
        data['hum'] = data['hum'].interpolate()
        days = data['date_hour'].dt.normalize()
        data = data[~days.isin(pd.to_datetime(outlier_days))]
        return data.reset_index(drop=True)


def add_calendar_features(data):
    """Voeg tijdselementen toe die uit 'date_hour' volgen."""
    data = data.copy()
    time = data['date_hour']
    data['hour'] = time.dt.hour
    data['weekday'] = time.dt.dayofweek           # 0 = maandag, 6 = zondag
    data['day_name'] = time.dt.day_name()
    data['week'] = time.dt.isocalendar().week.astype(int)
    data['month'] = time.dt.month
    data['year'] = time.dt.year - 2011            # 0 = 2011, 1 = 2012
    data['dayofyear'] = time.dt.dayofyear
    # werkdag = maandag t/m vrijdag en geen feestdag
    is_weekday = data['weekday'] < 5
    data['workingday'] = (is_weekday & (data['holiday'] == 0)).astype(int)
    return data


# Data inladen en opschonen zoals in Opdracht 1 (paragraaf 1.9)
OUTLIER_DAYS = ['2012-10-29', '2012-10-30']
loader = DataLoader(DATA_FOLDER)
train = loader.clean(loader.load('train.csv'), OUTLIER_DAYS)
test = loader.clean(loader.load('test.csv'), outlier_days=[])
print(f'train: {len(train)} rijen | test: {len(test)} rijen')

# 2. Opdracht 2: Time series feature engineering

In deze opdracht onderzoeken we stationariteit, trend, seizoenspatronen en cycli, maken we daarvan
features en meten we wat hun invloed is op de voorspellingen. Dit was een **iteratief proces**:
features toevoegen → model testen → nieuwe features proberen (2.8).

## 2.1 Evaluatiemethode

Om de invloed van features en modellen eerlijk te meten, leggen we eerst de evaluatie vast.

- **Niet husselen.** Bij tijdreeksen mag je de data niet willekeurig verdelen: dan traint het model
  op de toekomst en wordt de score te optimistisch (*data leakage*). Cross-validatie bij tijdreeksen
  werkt daarom anders dan gewone k-fold (notebook *Cross validation*; De Haagse Hogeschool,
  z.d.-a): elke fold traint op het **verleden** en valideert op de periode **erna**
  (`TimeSeriesSplit`; opdracht 11 van *Regression Time Series 2*, De Haagse Hogeschool, z.d.-d;
  *Week 11 application remarks*, De Haagse Hogeschool, z.d.-j; Holdgraf, z.d.).
- **Trainset en testset.** Net als in opdracht 3.1 van *Regression Time Series 1* en de klasse
  `RegressionModels` uit *Regression Time Series 3* (`train_test_split(shuffle=False)`; De Haagse
  Hogeschool, z.d.-c, z.d.-e) houden we de **laatste periode**, 19 dagen (12–30 november 2012),
  apart als **testset**. Die gebruiken we pas aan het eind, één keer per model. De hyperparameters
  kiezen we met cross-validatie op de trainset. Daarna trainen we het beste model opnieuw op álle
  data voor Kaggle (Radhakrishnan, 2023a).
- **Folds op de trainset.** Elke validatieperiode is **456 uur (19 dagen)**, net zo lang als de
  Kaggle-test:
    - 4 **herfst-folds** (`TimeSeriesSplit(n_splits=4, test_size=456)`): de laatste 2,5 maand vóór de
      testset;
    - 3 **winter-folds**: 1–19 december 2011, 9–27 januari 2012 en 6–24 februari 2012.

  Meer folds geven een betrouwbaarder gemiddelde, maar kosten meer rekentijd; met 7 folds zitten
  beide seizoenen er goed in.
- **Waarom winter-folds? (iteratie na Kaggle-feedback)** In onze eerste versie gebruikten we alleen
  herfst-folds. Op Kaggle bleek dat modellen op log-schaal (lineaire regressie, SARIMAX) veel beter
  scoorden dan onze validatie voorspelde (zie 4.3). December heeft lage aantallen en gedraagt zich
  anders dan de drukke herfst. Daarom valideren we nu ook op winterperiodes. De code voor de
  winter-folds komt niet in de lesstof voor (Claude, 2026. Prompt 2: Winter-folds voor CV).

**Metrics** (opdracht 2 van *Regression 1*; De Haagse Hogeschool, z.d.-b): $R^2$ (verklaarde
variantie), MAE (gemiddelde absolute fout),
MSE (gemiddelde gekwadrateerde fout) en **RMSE** $=\sqrt{\text{MSE}}$, de maatstaf van Kaggle:
$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$
De RMSE heeft dezelfde eenheid als `cnt` (fietsen per uur) en straft grote fouten zwaar af.

In [ ]:
# Bron: Week 10 – Regression Time Series 2, opdracht 11 (TimeSeriesSplit)
# Bron: Week 11 – application remarks (TimeSeriesSplit)
# Bron: Week 8 – Regression 1, opdracht 2 (R2, MAE, MSE en RMSE)
# Bron: Week 8 – Cross validation (cross_val_score)
# Bron: Week 9 – Regression Time Series 1, opdracht 3.1 (testset)
# Prompt 2: Winter-folds voor CV

WINTER_STARTS = ['2011-12-01', '2012-01-09', '2012-02-06']
N_AUTUMN = 4


def make_folds(data, n_autumn=N_AUTUMN, winter_starts=WINTER_STARTS):
    """Maak de folds: (trainingsrijen, validatierijen) per fold.

    Elke fold traint op alle rijen vóór de validatieperiode van 456 uur.
    """
    autumn = TimeSeriesSplit(n_splits=n_autumn, test_size=HORIZON)
    folds = list(autumn.split(data))
    for start in pd.to_datetime(winter_starts):
        end = start + pd.Timedelta(hours=HORIZON)
        train_idx = np.where(data['date_hour'] < start)[0]
        val_idx = np.where((data['date_hour'] >= start)
                           & (data['date_hour'] < end))[0]
        folds.append((train_idx, val_idx))
    return folds


def rmse(y_true, y_pred):
    """Root Mean Squared Error (de maatstaf van Kaggle)."""
    return np.sqrt(mean_squared_error(y_true, y_pred))


def regression_metrics(y_true, y_pred):
    """R2, MAE, MSE en RMSE (zoals in de les Regression 1)."""
    mse = mean_squared_error(y_true, y_pred)
    return {'R2': r2_score(y_true, y_pred),
            'MAE': mean_absolute_error(y_true, y_pred),
            'MSE': mse, 'RMSE': np.sqrt(mse)}


# Trainset en testset: de laatste 19 dagen zijn de testset (holdout)
train_part = train[train['date_hour'] < HOLDOUT_START].reset_index(drop=True)
holdout = train[train['date_hour'] >= HOLDOUT_START].reset_index(drop=True)
CV = make_folds(train_part)


def cv_rmse(model, X, y):
    """Gemiddelde RMSE over alle folds van de trainset."""
    scores = cross_val_score(model, X, y, cv=CV,
                             scoring='neg_root_mean_squared_error')
    return -scores.mean()


fold_overview = []
for fold, (train_idx, val_idx) in enumerate(CV, start=1):
    fold_overview.append({
        'fold': fold,
        'type': 'herfst' if fold <= N_AUTUMN else 'winter',
        'training t/m': train_part['date_hour'][train_idx].max(),
        'validatie van': train_part['date_hour'][val_idx].min(),
        'validatie t/m': train_part['date_hour'][val_idx].max()})
print(f'trainset: {len(train_part)} uur | testset: '
      f'{holdout["date_hour"].min():%d-%m-%Y} t/m '
      f'{holdout["date_hour"].max():%d-%m-%Y} ({len(holdout)} uur)')
pd.DataFrame(fold_overview)

## 2.2 Stationariteit (Augmented Dickey-Fuller-test)

Een tijdreeks is **stationair** als het gemiddelde, de variantie en de autocorrelatie in de tijd
constant zijn (Radhakrishnan, 2023d). Modellen als ARMA vereisen dat. We testen dit met de
**Augmented Dickey-Fuller-test**, zoals in opdracht 5 van *Time series analysis 3* (De Haagse
Hogeschool, z.d.-h):

- nulhypothese $H_0$: de reeks is **niet** stationair;
- p-waarde < 0,05 → $H_0$ verwerpen → de reeks is stationair.

Een niet-stationaire reeks kun je stationair maken door te **differentiëren**: $y_t - y_{t-1}$. Dat is
de *I* (integrated) in ARIMA.

In [ ]:
# Bron: Week 11 – Time series analysis 3, opdracht 5 (adfuller)
# Bron: Week 11 – slides Timeseries analyse 3 (ADF, differentiëren)

# Regelmatige reeksen: per uur (ontbrekende uren geïnterpoleerd) en per dag
hourly = train.set_index('date_hour')['cnt'].asfreq('h').interpolate()
daily = (train.set_index('date_hour')['cnt'].resample('D').mean()
         .asfreq('D').interpolate())


def adf_test(series, name):
    """Augmented Dickey-Fuller-test (H0: de reeks is niet stationair)."""
    statistic, p_value = adfuller(series.dropna(), autolag='AIC')[:2]
    return {'reeks': name, 'ADF-statistiek': round(statistic, 2),
            'p-waarde': round(p_value, 4),
            'stationair (p < 0,05)': p_value < 0.05}


pd.DataFrame([adf_test(hourly, 'cnt per uur'),
              adf_test(daily, 'cnt per dag (gemiddelde)'),
              adf_test(daily.diff(), 'cnt per dag, gedifferentieerd')])

**Toelichting.**

- **Per uur** is de reeks volgens de ADF-test stationair (p < 0,0001): het sterke dagritme trekt de
  reeks steeds terug naar hetzelfde niveau.
- **Per dag** is de reeks **niet** stationair (p = 0,29): het gemiddelde verandert door de trend en
  het jaarseizoen.
- **Na differentiëren** is de dagreeks wel stationair (p < 0,0001).
- Gevolgen: (1) een ARIMA-model op dagniveau zou $d = 1$ nodig hebben; (2) voor SARIMAX op uurniveau
  over een korte periode (8 weken) nemen we $d = 0$ (3.8); (3) de machine-learning modellen krijgen de
  trend als feature (2.3).

## 2.3 Trend

Een **trend** is een langzame verandering van het niveau over een lange periode. We onderzoeken
de trend op twee manieren uit de lessen:

1. **Voortschrijdende gemiddelden** (*moving averages*) met verschillende window-groottes. Een
   window zo groot als een seizoensperiode (7 dagen, 365 dagen) middelt dat seizoen weg
   (Radhakrishnan, 2023b; Kaggle-cursus les 2, Holbrook, z.d.-f).
2. **`DeterministicProcess`** (Kaggle-cursus les 1 en 2, Holbrook, z.d.-c, z.d.-f; opdracht 5–7
   van *Regression Time Series 2*, De Haagse Hogeschool, z.d.-d; *Week 10 application remarks*,
   De Haagse Hogeschool, z.d.-i): een tijd-dummy $t$ (en $t^2$, $t^3$ bij een hogere *order*) als
   feature voor lineaire regressie, $y(t) = a + b\,t + c\,t^2 + \dots$ Omdat het een functie van de
   tijd is, kun je hiermee **extrapoleren** naar de toekomst.

In [ ]:
# Bron: Kaggle Time Series, les 2 – Trend (Holbrook)
# Bron: Week 9 – Time series analysis 1, opdracht 2 en 3

fig, ax = plt.subplots()
ax.plot(daily, alpha=0.3, color='grey', label='gemiddelde per dag')
for window in [7, 30, 365]:
    moving_average = daily.rolling(window, center=True,
                                   min_periods=window // 2).mean()
    ax.plot(moving_average, label=f'voortschrijdend gemiddelde ({window} d)')
ax.set(title='Trend met voortschrijdende gemiddelden', ylabel='cnt per uur')
ax.legend()
plt.show()

# Groei: vergelijk dezelfde maanden (januari t/m november)
same_months = add_calendar_features(train).query('month <= 11')
mean_per_year = same_months.groupby('year')['cnt'].mean()
growth = (mean_per_year[1] / mean_per_year[0] - 1) * 100
print(f'Gemiddelde jan-nov 2011: {mean_per_year[0]:.0f} | '
      f'2012: {mean_per_year[1]:.0f} | groei: +{growth:.0f}%')

Welke **order** van de trend past het best? Net als in opdracht 5–7 van *Regression Time Series 2*
(De Haagse Hogeschool, z.d.-d) fitten we lineaire regressie met `DeterministicProcess` voor
order 1, 2 en 3, hier op het gemiddelde per dag en samen met Fourier-termen voor het jaarseizoen
(zie 2.4; Kaggle-cursus les 3, Holbrook, z.d.-d). We trainen op de trainset en extrapoleren met
`out_of_sample` naar de testset en de Kaggle-periode (t/m 19 december), zoals in Kaggle-cursus
les 2 (Holbrook, z.d.-f).

In [ ]:
# Bron: Week 10 – Regression Time Series 2, opdracht 5, 6 en 7
# Bron: Kaggle Time Series, les 2 – Trend (Holbrook)
# Bron: Kaggle Time Series, les 3 – Seasonality (Holbrook)

daily_train = daily[daily.index < HOLDOUT_START]
daily_test = daily[daily.index >= HOLDOUT_START]
future_days = pd.date_range(HOLDOUT_START, '2012-12-19', freq='D')

trend_rows = []
fig, ax = plt.subplots()
ax.plot(daily, color='grey', alpha=0.4, label='werkelijk (gemiddelde per dag)')
for order in [1, 2, 3]:
    dp = DeterministicProcess(index=daily_train.index, constant=False,
                              order=order, drop=True,
                              additional_terms=[CalendarFourier('YE', 3)])
    X_in = dp.in_sample()
    X_out = dp.out_of_sample(steps=len(future_days),
                             forecast_index=future_days)
    trend_model = Pipeline([('scale', StandardScaler()),
                            ('lr', LinearRegression())])
    trend_model.fit(X_in, daily_train)
    fitted = pd.Series(trend_model.predict(X_in), index=X_in.index)
    forecast = pd.Series(trend_model.predict(X_out), index=future_days)
    trend_rows.append({
        'order': order,
        'R2 trainset': r2_score(daily_train, fitted),
        'RMSE testset (per dag)': rmse(daily_test,
                                       forecast.loc[daily_test.index]),
        'voorspelling 19 dec': forecast.iloc[-1]})
    ax.plot(pd.concat([fitted, forecast]), label=f'order {order}')
ax.axvline(HOLDOUT_START, color='black', linestyle='--', label='start testset')
ax.set(title='Trend (DeterministicProcess) + jaarseizoen, met extrapolatie',
       ylabel='cnt per uur')
ax.legend()
plt.show()
pd.DataFrame(trend_rows).round(2)

**Conclusie trend.**

- De voortschrijdende gemiddelden laten een duidelijke **stijgende trend** zien: in 2012 gemiddeld
  **+65%** verhuur ten opzichte van dezelfde maanden in 2011. Het 7-daagse gemiddelde volgt nog het
  weer, het 30-daagse het jaarseizoen; het 365-daagse gemiddelde stijgt geleidelijk.
- Alle drie de ordes passen even goed op de trainset ($R^2 \approx 0{,}78$), maar **order 1**
  voorspelt de testset het best (RMSE 59 per dag, tegen 63–66). Hogere ordes geven een gebogen
  trend die bij extrapolatie sneller de verkeerde kant op gaat. We kiezen dus **order 1**: de kolom
  `trend` uit `DeterministicProcess`.
- De Fourier-termen voor het jaarseizoen zorgen ervoor dat de lijn in de winter daalt. Een trend
  alleen kan dat niet (2.4).
- **Bomen kunnen een trend niet extrapoleren** (*Week 10 application remarks*, De Haagse Hogeschool,
  z.d.-i; Kaggle-cursus les 5, Holbrook, z.d.-b). Voor bomen gebruiken we daarom
  `year` (niveau per jaar); voor lineaire regressie en het hybride model de doorlopende `trend`.

## 2.4 Seizoenspatronen met Fourier-analyse

Elke periodieke tijdreeks is een som van sinussen en cosinussen (Radhakrishnan, 2023c):
$$f(t) = \frac{a_0}{2} + \sum_{k=1}^{\infty}\big(a_k\cos(2\pi k t) + b_k\sin(2\pi k t)\big)$$
Met de **Fast Fourier Transform** (FFT) zetten we de uurreeks om naar het **frequentiedomein**. Net
als in het notebook *Time series analysis 2* (De Haagse Hogeschool, z.d.-g) en opdracht 1 en 2 van
*Regression Time Series 3* (De Haagse Hogeschool, z.d.-e):

- amplitude $= \frac{2}{N}|\text{FFT}|$ (eerste helft van de frequenties);
- de frequentie-as is in eenheden $1/N$: index $k$ hoort bij frequentie $k/N$ (per uur);
- de **duur** (periode) is $1/\text{frequentie}$;
- met `find_peaks` en een grenswaarde voor de amplitude vinden we de seizoenspatronen.

We trekken eerst het gemiddelde af, zodat de piek bij frequentie 0 (de constante) verdwijnt. In de
tweede grafiek staat hetzelfde spectrum met de periode op een log-schaal, zoals het *periodogram*
in Kaggle-cursus les 3 (Holbrook, z.d.-d; *Week 10 application remarks*, De Haagse Hogeschool,
z.d.-i).

In [ ]:
# Bron: Week 10 – Time series analysis 2 (FFT, amplitude, find_peaks)
# Bron: Week 12 – Regression Time Series 3, opdracht 1 en 2
# Bron: Kaggle Time Series, les 3 – Seasonality (Holbrook):
#       periodogram met een log-schaal

values = hourly.values - hourly.values.mean()
N = len(values)
magnitude = 2.0 / N * np.abs(fft(values)[:N // 2])
frequency = np.arange(N // 2) / N                  # cycli per uur

peaks, _ = find_peaks(magnitude, height=20)        # grenswaarde amplitude 20
spectrum = pd.DataFrame({'amplitude': magnitude[peaks],
                         'frequentie (per uur)': frequency[peaks],
                         'duur (uren)': 1 / frequency[peaks]})
spectrum['duur (dagen)'] = spectrum['duur (uren)'] / 24

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
axes[0].plot(frequency, magnitude)
axes[0].plot(frequency[peaks], magnitude[peaks], 'r.', markersize=8)
axes[0].set(xlim=(0, 0.25), title='Frequentiedomein (FFT) met pieken',
            xlabel='frequentie (cycli per uur)', ylabel='amplitude')
# Zelfde spectrum als periodogram: periode op een log-schaal
axes[1].semilogx(1 / frequency[1:] / 24, magnitude[1:])
for days, label in [(0.5, '12 uur'), (1, 'dag'), (7, 'week'), (365, 'jaar')]:
    axes[1].axvline(days, color='grey', linestyle='--')
    axes[1].text(days, magnitude.max() * 0.9, label, rotation=90)
axes[1].set(title='Zelfde spectrum, periode op log-schaal',
            xlabel='periode (dagen)', ylabel='amplitude')
plt.tight_layout()
plt.show()
spectrum.sort_values('amplitude', ascending=False).round(3)

**Conclusie Fourier-analyse** (grenswaarde amplitude = 20):

- De sterkste piek ligt bij **24 uur** (amplitude 150): het dagpatroon.
- De pieken bij **12, 8, 6 en 4,8 uur** zijn **harmonischen**: veelvouden van de basisfrequentie 1/24
  (Radhakrishnan, 2023c). Ze ontstaan omdat het dagpatroon geen nette sinus is (twee spitsen). Voor
  het dagpatroon zijn dus veel Fourier-termen nodig, of dummies per uur.
- **Weekpatroon:** er is geen losse piek bij 168 uur. Het weekpatroon zit in de **zijbanden** naast
  de dag-pieken: 12,9 en 11,2 uur zijn precies $\frac{1}{12} \pm \frac{1}{168}$, en 14 en 10,5 uur
  zijn $\frac{1}{12} \pm \frac{2}{168}$. Het weekpatroon verandert dus de **vorm** van het dagpatroon
  (werkdag vs. weekend), net als in de decompositie (1.7).
- **Jaarpatroon:** pieken bij 8400 uur (350 dagen ≈ een jaar) en 4200 uur (≈ een half jaar). Met
  2 jaar data is de resolutie grof. De piek bij 23,9 uur is $\frac{1}{24} + \frac{1}{8400}$: de
  dagpiek is in de zomer hoger dan in de winter.
- **Features:** dag → dummies voor uur × werkdag (2.8); week → `workingday` en `weekday`; jaar →
  Fourier-termen met `CalendarFourier` (lineaire modellen) of `dayofyear` (bomen).

### Seizoen als dummies of als Fourier-termen?

Een seizoen kun je op twee manieren als feature geven (Kaggle-cursus les 3, Holbrook, z.d.-d;
*Week 10 application remarks*, De Haagse Hogeschool, z.d.-i):

- **Seizoensdummies** (one-hot): één kolom per uur. Eén categorie laat je weg, omdat die al in de
  constante zit (anders zijn de kolommen samen perfect gecorreleerd: de *dummy trap*). Dummies kunnen
  elke vorm van het patroon volgen, maar kosten veel kolommen.
- **Fourier-termen** (`CalendarFourier`): een paar sin/cos-paren. Weinig kolommen, maar een gladde
  vorm. Handig voor lange seizoenen (het jaar) met weinig data per periode.

`DeterministicProcess(seasonal=True)` maakt de dummies. Met `drop=True` worden kolommen die perfect
afhangen van andere kolommen verwijderd. Hieronder zie je wat er gebeurt als je dummies én Fourier-termen
voor dezelfde periode toevoegt (opdracht 3 uit *Regression Time Series 3*; De Haagse Hogeschool,
z.d.-e):

In [ ]:
# Bron: Week 12 – Regression Time Series 3, opdracht 3 (seasonal en drop)
# Bron: Kaggle Time Series, les 3 – Seasonality (Holbrook)

week_index = pd.date_range('2012-01-02', periods=7 * 24, freq='h')
fourier_day = CalendarFourier(freq='D', order=2)
settings = {
    'uur-dummies (seasonal=True)': dict(seasonal=True),
    'uur-dummies + Fourier dag, drop=True': dict(
        seasonal=True, additional_terms=[fourier_day], drop=True),
    'uur-dummies + Fourier dag, drop=False': dict(
        seasonal=True, additional_terms=[fourier_day], drop=False),
}
for name, setting in settings.items():
    dp = DeterministicProcess(index=week_index, constant=True, **setting)
    columns = list(dp.in_sample().columns)
    print(f'{name}: {len(columns)} kolommen, bijv. {columns[:3]} ... '
          f'{columns[-2:]}')

Met `seasonal=True` maakt `DeterministicProcess` 23 uur-dummies (`s(2,24)` t/m `s(24,24)`): het
eerste uur zit in de constante. Voeg je ook Fourier-termen voor de dag toe, dan verwijdert
`drop=True` die: ze bevatten geen nieuwe informatie, want de 23 dummies beschrijven al elk mogelijk
dagpatroon. Met `drop=False` blijven ze staan en zijn de kolommen perfect afhankelijk
(multicollineair). Daarom kies je voor één seizoen óf dummies óf Fourier-termen. Welke keuze het best
werkt, onderzoeken we in 2.8.

Voor het **jaarseizoen** gebruiken we Fourier-termen: met 2 jaar data hebben we per dag van het jaar
maar 2 metingen, te weinig voor dummies. De functie hieronder maakt met `DeterministicProcess` en
`CalendarFourier` de trend en de Fourier-termen voor **elk uur van 1-1-2011 t/m 19-12-2012** (train
én test), zoals in Kaggle-cursus les 3 (Holbrook, z.d.-d). Dit mag, want deze termen hangen alleen
van de tijd af en niet van `cnt`. Daarna tekenen we de Fourier-termen, zoals in opdracht 4 van
*Regression Time Series 3* (De Haagse Hogeschool, z.d.-e). De hulpfunctie `short_name` (kortere
kolomnamen) komt niet in de lesstof voor (Claude, 2026. Prompt 3: Functie short_name).

In [ ]:
# Bron: Kaggle Time Series, les 3 – Seasonality (Holbrook)
# Bron: Week 12 – Regression Time Series 3, opdracht 3 en 4
# Prompt 3: Functie short_name

def short_name(column):
    """Korte kolomnaam, bijv. 'sin(1,freq=YE-DEC)' -> 'sin_y1'."""
    match = re.match(r'(sin|cos)\((\d+),freq=([A-Z])', column)
    if match is None:
        return column
    function, k, freq = match.groups()
    period = 'y' if freq == 'Y' else 'd'
    return f'{function}_{period}{k}'


def make_deterministic_terms(yearly_order=3, daily_order=0, trend_order=1):
    """Trend + Fourier-termen (DeterministicProcess) voor elk uur."""
    index = pd.date_range('2011-01-01', '2012-12-19 23:00', freq='h')
    terms = []
    if yearly_order > 0:
        terms.append(CalendarFourier(freq='YE', order=yearly_order))
    if daily_order > 0:
        terms.append(CalendarFourier(freq='D', order=daily_order))
    dp = DeterministicProcess(index=index, constant=False, order=trend_order,
                              additional_terms=terms, drop=True)
    terms_df = dp.in_sample()
    terms_df.columns = [short_name(column) for column in terms_df.columns]
    return terms_df


deterministic_example = make_deterministic_terms(yearly_order=3)
print(list(deterministic_example.columns))
(deterministic_example[['sin_y1', 'cos_y1', 'sin_y2', 'sin_y3']]
 .resample('D').first()
 .plot(subplots=True, figsize=(12, 6),
       title='Fourier-termen voor het jaarseizoen (orde 1, 2 en 3)'))
plt.show()

Elke orde $k$ voegt een sinus en een cosinus toe die $k$ keer per jaar op en neer gaan. Met meer
termen kan het model een fijnere vorm van het jaarseizoen volgen, maar het risico op overfitting
groeit. De beste orde onderzoeken we in 2.8.

## 2.5 Cycli, autocorrelatie en lags

Bij **serial dependence** hangt $y(t)$ af van eerdere waarden: $y(t) = a\cdot y(t-1) + b\cdot
y(t-2) + \dots$ (Kaggle-cursus les 4, Holbrook, z.d.-e; *Week 10 application remarks*, De Haagse
Hogeschool, z.d.-i). De **ACF** toont de correlatie met lag $k$:
$$\rho_k = \frac{\text{Cov}(y_t, y_{t-k})}{\text{Var}(y_t)}$$
Omdat lag 2 ook via lag 1 samenhangt, laat de **PACF** per lag alleen de *nieuwe* correlatie zien.
Lags buiten het blauwe gebied zijn bruikbaar als feature (Radhakrishnan, 2023d). We maken de plots
zoals in opdracht 6 van *Time series analysis 3* en opdracht 7 van *Regression Time Series 3* (De
Haagse Hogeschool, z.d.-h, z.d.-e).

In [ ]:
# Bron: Week 12 – Regression Time Series 3, opdracht 7 (plot_pacf)
# Bron: Week 11 – Time series analysis 3, opdracht 6 (ACF en PACF)
# Bron: Kaggle Time Series, les 4 – Time Series as Features (Holbrook)

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
plot_acf(hourly, lags=24 * 8, ax=axes[0], title='ACF (8 dagen)')
plot_pacf(hourly, lags=48, ax=axes[1], method='ywm', title='PACF (2 dagen)')
for ax in axes:
    ax.set_xlabel('lag (uren)')
plt.show()

lags = {'1 uur': 1, '2 uur': 2, '1 dag (24)': 24, '1 week (168)': 168,
        '3 weken (504)': 504}
pd.Series({name: hourly.autocorr(lag=lag) for name, lag in lags.items()},
          name='autocorrelatie').round(3)

**Conclusie cycli en autocorrelatie.**

- De ACF golft met een **periode van 24 uur** en is extra hoog bij **168 uur** (een week, 0,87):
  dezelfde cycli als in de Fourier-analyse.
- De autocorrelatie neemt **geleidelijk** af (met golven). Volgens de les past dan een **AR-model**;
  bij een MA-model valt de ACF na enkele lags plotseling weg (Radhakrishnan, 2023d).
- De **PACF** heeft grote waarden bij lag 1 (positief), lag 2 (negatief) en rond lag 24. Voor een
  lag-model kiezen we daarom lags **1, 2, 3 en 24**; voor SARIMAX een AR-orde van 1–2 met eventueel
  een seizoens-AR bij $s = 24$ (3.8).

### Voorspellen met lags

We maken lag-features met de functie `make_lags` uit Kaggle-cursus les 4 (Holbrook, z.d.-e), zoals
gevraagd in opdracht 8 van *Regression Time Series 3* (De Haagse Hogeschool, z.d.-e), en trainen
een XGBoost-model met lags 1, 2, 3 en 24 (gekozen op basis van de PACF) plus tijd en weer. Daarna
voorspellen we de testset (12–30 november) op twee manieren (opdracht 9 van *Regression Time
Series 3*):

1. **1 uur vooruit**: met de *echte* vorige waarden als lags (kan alleen als je elk uur de echte
   verhuur kent);
2. **recursief 456 uur vooruit**: zoals bij Kaggle, waar `cnt` in de testperiode onbekend is. Elke
   voorspelling wordt dan de lag van de volgende uren. Dit is de *recursieve strategie* uit
   Kaggle-cursus les 6 (Holbrook, z.d.-a). Die les geeft geen code voor deze strategie; de functie
   `recursive_forecast` is gemaakt met hulp van GenAI (Claude, 2026. Prompt 4: Recursief voorspellen met lags).

In [ ]:
# Bron: Kaggle Time Series, les 4 – Time Series as Features (Holbrook)
# Bron: Week 12 – Regression Time Series 3, opdracht 8 en 9
# Bron: Kaggle Time Series, les 6 – Forecasting With Machine Learning
#       (Holbrook): idee van de recursieve strategie
# Prompt 4: Recursief voorspellen met lags

def make_lags(ts, lags):
    """Maak lag-kolommen van een reeks (naar Holbrook, z.d.-e)."""
    return pd.concat({f'cnt_lag_{lag}': ts.shift(lag) for lag in lags},
                     axis=1)


def recursive_forecast(model, history, future, lags):
    """Voorspel uur na uur; elke voorspelling wordt een lag van later."""
    history = history.copy()
    predictions = []
    for time_stamp, row in future.iterrows():
        row = row.copy()
        for lag in lags:
            row[f'cnt_lag_{lag}'] = history[time_stamp
                                            - pd.Timedelta(hours=lag)]
        prediction = max(float(model.predict(row.to_frame().T)[0]), 0.0)
        predictions.append(prediction)
        history[time_stamp] = prediction
    return pd.Series(predictions, index=future.index)


# Regelmatige uurreeks met weer en tijdselementen
regular = train.set_index('date_hour').asfreq('h')
regular[['holiday', 'weathersit']] = regular[['holiday',
                                              'weathersit']].ffill()
regular = add_calendar_features(regular.interpolate().reset_index())
regular = regular.set_index('date_hour')

LAGS = [1, 2, 3, 24]
LAG_BASE = ['hour', 'weekday', 'workingday', 'holiday', 'weathersit', 'temp',
            'hum', 'windspeed']
lag_data = pd.concat([regular[LAG_BASE], make_lags(regular['cnt'], LAGS)],
                     axis=1)
is_train = lag_data.index < HOLDOUT_START
X_lag_train = lag_data[is_train].dropna()
y_lag_train = regular.loc[X_lag_train.index, 'cnt']
X_lag_test = lag_data[~is_train]
y_lag_test = regular.loc[X_lag_test.index, 'cnt']

lag_model = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                         subsample=0.8, random_state=RANDOM_STATE)
lag_model.fit(X_lag_train, y_lag_train)
one_step = lag_model.predict(X_lag_test)
recursive = recursive_forecast(lag_model, regular['cnt'][is_train],
                               X_lag_test, LAGS)

lag_results = {'1 uur vooruit (echte lags)': rmse(y_lag_test, one_step)}
for days in [1, 3, 7, 19]:
    hours = days * 24
    lag_results[f'recursief, eerste {days} dag(en)'] = rmse(
        y_lag_test[:hours], recursive[:hours])
display(pd.Series(lag_results, name='RMSE testset').round(1))

week = slice(0, 7 * 24)
plt.plot(y_lag_test.index[week], y_lag_test.values[week], label='werkelijk')
plt.plot(y_lag_test.index[week], one_step[week], label='1 uur vooruit')
plt.plot(y_lag_test.index[week], recursive.values[week],
         label='recursief (456 uur vooruit)')
plt.title('Voorspellen met lags: 1 uur vooruit vs recursief')
plt.legend()
plt.show()

**Conclusie: voorspellen met lags.**

- Met de **echte** vorige waarden (1 uur vooruit) is de RMSE maar **34**: beter dan al onze andere
  modellen. Maar dat kan alleen als je elk uur de echte verhuur van het uur ervoor kent.
- Bij Kaggle moeten we **19 dagen vooruit** voorspellen zonder `cnt`. Recursief voorspellen geeft dan
  een RMSE van **92**: elke fout wordt de input van het volgende uur en **stapelt zich op**
  (voortplanting van fouten; Radhakrishnan, 2023d). Op de eerste dag (een feestdag, 12 november)
  gaat het al flink mis (123).
- Ons beste model zonder lags haalt op dezelfde testset een RMSE van ±51 (3.10).
- **Belangrijkste nadeel van lags** (de vraag uit opdracht 9 van *Regression Time Series 3*; De
  Haagse Hogeschool, z.d.-e): je hebt recente echte waarden nodig, en die zijn er bij een
  lange horizon niet. Daarom gebruiken we geen korte lags. Een lag van minstens 19 dagen
  (`level_lag21`) testen we wel (2.8).

## 2.6 Overige tijdgebaseerde features

Naast trend, seizoenen en cycli noemt de les ook **opmerkelijke dagen** en specifieke
tijdsperiodes als features (Radhakrishnan, 2023d):

- `holiday` en `workingday`: feestdagen en werkdagen. Het dagpatroon hangt sterk af van het type
  dag (EDA).
- `hour_wd`: combinatie van uur en werkdag (bijv. `8_1` = 8 uur op een werkdag). Bomen vinden zo'n
  interactie zelf; een lineair model heeft hem als dummies nodig.
- `dayofyear`, `month`, `year`: positie in het jaar en het niveau per jaar (voor bomen).
- `level_lag21`: gemiddelde verhuur per uur in de week van **21 dagen geleden**. Dit is een lag die
  wél beschikbaar is in de hele testperiode (horizon 19 dagen): de *lead time* is groter dan de
  horizon (Kaggle-cursus les 6, Holbrook, z.d.-a).
- `weather_prev1` en `weather_next1`: de weersituatie van het **vorige** en het **volgende** uur.
  Na een regenbui zijn de straten nat, en als er over een uur regen komt, stappen minder mensen op.
  Het weer is voor alle testuren bekend, dus dit is geen data leakage (we gebruiken geen `cnt`).
  Het idee om lags van een **andere** reeks te gebruiken komt uit Kaggle-cursus les 4 (*leading
  indicators*; Holbrook, z.d.-e). Deze feature hebben we in de tweede iteratie toegevoegd.

## 2.7 Klasse `FeatureEngineer`

De klasse `FeatureEngineer` maakt alle features op dezelfde manier voor de trainingsdata en de
Kaggle-testdata. Net als in opdracht 9 van *Regression Time Series 2* (De Haagse Hogeschool,
z.d.-d) voegen methods van een klasse de tijdreeks-features toe. De klasse zelf komt niet in de
lesstof voor (Claude, 2026. Prompt 5: Klasse FeatureEngineer).

In [ ]:
# Bron: Week 10 – Regression Time Series 2, opdracht 9 (features met
#       methods van een klasse toevoegen)
# Bron: Kaggle Time Series, les 4 (lags van een andere reeks) en les 6
#       (lead time: de lag is in de hele horizon bekend) (Holbrook)
# Prompt 5: Klasse FeatureEngineer

class FeatureEngineer:
    """Maakt alle tijdreeks-features voor de modellen."""

    def __init__(self, history, future):
        # history = trainingsdata met 'cnt'; nodig voor de lag-feature
        self.history = history.set_index('date_hour')['cnt']
        # weersituatie van alle uren (train + test), zonder gaten
        weather = pd.concat([history, future]).set_index('date_hour')
        self.weather = weather['weathersit'].sort_index().asfreq('h').ffill()

    @staticmethod
    def add_deterministic(data, yearly_order, daily_order):
        """Trend en Fourier-termen uit DeterministicProcess (zie 2.4)."""
        terms = make_deterministic_terms(yearly_order, daily_order)
        terms = terms.reindex(data['date_hour'])
        for column in terms.columns:
            data[column] = terms[column].to_numpy()
        return data

    def add_level_lag(self, data, days=21, window=7):
        """Gemiddelde verhuur per uur in de week van 'days' dagen geleden."""
        daily_level = self.history.resample('D').mean()
        level = daily_level.rolling(window, min_periods=4).mean()
        past_day = data['date_hour'].dt.normalize() - pd.Timedelta(days=days)
        data['level_lag21'] = past_day.map(level).to_numpy()
        return data

    def add_weather_neighbours(self, data):
        """Weersituatie van het vorige en het volgende uur."""
        previous = self.weather.shift(1).bfill()
        following = self.weather.shift(-1).ffill()
        data['weather_prev1'] = data['date_hour'].map(previous).to_numpy()
        data['weather_next1'] = data['date_hour'].map(following).to_numpy()
        return data

    def transform(self, data, yearly_order=6, daily_order=4):
        """Voeg alle features toe en geef een nieuwe DataFrame terug."""
        data = add_calendar_features(data)
        data['hour_wd'] = (data['hour'].astype(str) + '_'
                           + data['workingday'].astype(str))
        data = self.add_deterministic(data, yearly_order, daily_order)
        data = self.add_level_lag(data)
        data = self.add_weather_neighbours(data)
        return data


engineer = FeatureEngineer(train, test)
train_fe = engineer.transform(train)
test_fe = engineer.transform(test)
train_fe_part = train_fe[train_fe['date_hour'] < HOLDOUT_START]
train_fe_part = train_fe_part.reset_index(drop=True)
print('aantal kolommen:', train_fe.shape[1])
train_fe.tail(3)

## 2.8 Invloed van de features op de voorspelling

**Stap 1 — welke features helpen?** Net als in opdracht 5 van *Regression Time Series 3* (De Haagse
Hogeschool, z.d.-e) onderzoeken we de invloed van features op de voorspellingen. We testen
feature-sets met **hetzelfde model** (XGBoost met vaste instellingen) en dezelfde cross-validatie.
Zo zien we alleen het effect van de features. We bouwen stap voor stap op. De functie
`compare_feature_sets` komt niet in de lesstof voor (Claude, 2026. Prompt 6: Feature-sets vergelijken).

In [ ]:
# Bron: Week 12 – Regression Time Series 3, opdracht 5 (invloed features)
# Prompt 6: Feature-sets vergelijken

def compare_feature_sets(data, feature_sets, log_target=False):
    """Vergelijk feature-sets met hetzelfde XGBoost-model (CV)."""
    rows = []
    for name, features in feature_sets.items():
        model = XGBRegressor(n_estimators=800, learning_rate=0.03,
                             max_depth=7, subsample=0.8,
                             colsample_bytree=0.8,
                             random_state=RANDOM_STATE)
        if log_target:
            # model leert log(1 + cnt) en zet de voorspelling terug
            model = TransformedTargetRegressor(model, func=np.log1p,
                                               inverse_func=np.expm1)
        rows.append({'feature-set': name, 'aantal features': len(features),
                     'log-target': log_target,
                     'CV-RMSE': cv_rmse(model, data[features], data['cnt'])})
    return pd.DataFrame(rows)


WEATHER = ['holiday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed']
DAY_WEEK = ['hour', 'weekday', 'workingday']
WEATHER_NEXT = ['weather_prev1', 'weather_next1']
BEST_SET = WEATHER + DAY_WEEK + ['year', 'dayofyear']
FOURIER_3 = ['sin_y1', 'cos_y1', 'sin_y2', 'cos_y2', 'sin_y3', 'cos_y3']

feature_sets = {
    '1. alleen weer': WEATHER,
    '2. + dag- en weekseizoen': WEATHER + DAY_WEEK,
    '3. + trend (year)': WEATHER + DAY_WEEK + ['year'],
    '4. + trend + month': WEATHER + DAY_WEEK + ['year', 'month'],
    '5. + trend + dayofyear': BEST_SET,
    '6. + trend (DeterministicProcess) + Fourier jaar':
        WEATHER + DAY_WEEK + ['trend'] + FOURIER_3,
    '7. set 5 + level_lag21': BEST_SET + ['level_lag21'],
    '8. set 5 + weer vorig/volgend uur': BEST_SET + WEATHER_NEXT,
}
feature_results = compare_feature_sets(train_fe_part, feature_sets)
log_result = compare_feature_sets(
    train_fe_part, {'9. set 8 met log(cnt)': BEST_SET + WEATHER_NEXT},
    log_target=True)
pd.concat([feature_results, log_result], ignore_index=True).round(1)

**Conclusie feature-onderzoek** (XGBoost, CV-RMSE):

- **Alleen weer:** 182. Het **dag- en weekseizoen** (`hour`, `weekday`, `workingday`) halveert de
  fout (94): het dagpatroon is veruit het belangrijkst.
- **Trend** (`year`): 63. Zonder trend kan het model het hogere niveau van 2012 niet voorspellen.
- **Jaarseizoen:** `month` (50,1) en `dayofyear` (50,8) geven nog een grote verbetering en scoren
  bijna gelijk. We kiezen `dayofyear`, omdat de hele Kaggle-testperiode in december valt: `month` is
  daar steeds 12, `dayofyear` maakt wel onderscheid tussen begin en midden december.
- `trend` + Fourier-termen (51,2) werkt voor XGBoost ongeveer even goed. Voor lineaire modellen zijn
  juist deze features nodig (stap 2).
- **`level_lag21`** maakt het slechter (55,5). De hoge autocorrelatie bij 504 uur komt vooral door het
  vaste dag- en weekpatroon, en dat kent het model al. Het niveau van 3 weken geleden voegt vooral
  ruis toe (bijv. het weer van toen). We gebruiken deze feature niet.
- **Weer van het vorige en volgende uur** (iteratie 2) verbetert de score van 50,8 naar **47,8**.
- Een **log-transformatie** helpt XGBoost niet (53,0): bomen gaan zelf goed om met de scheve
  verdeling, en terugrekenen geeft de mediaan, waardoor de drukke uren te laag worden voorspeld. In
  3.10 lossen we dit beter op met Poisson-loss.

**Stap 2 — welke Fourier-orde en welk dagpatroon voor het lineaire model?** Opdracht 6 uit
*Regression Time Series 3* (De Haagse Hogeschool, z.d.-e): helpen de zwakkere seizoenspatronen (een
hogere Fourier-orde)? We testen dit met lineaire regressie op $\log(1+\text{cnt})$ (uitleg in 3.3),
met de trend, het weer en verschillende ordes voor het jaarseizoen. Daarna vergelijken we drie
manieren om het dagpatroon te geven: uur-dummies, dummies voor uur × werkdag, of Fourier-termen voor
de dag. Categorieën worden dummies met één categorie weggelaten (opdracht 1 van *Regression Time
Series 2*; *Week 10 application remarks*; De Haagse Hogeschool, z.d.-d, z.d.-i) en alleen de
numerieke features worden gestandaardiseerd (Radhakrishnan, 2025). De pipeline in de functie
`linear_model` (`ColumnTransformer` en log-target) komt niet in de lesstof voor (Claude, 2026.
Prompt 7: Lineair model met dummies en log-target).

In [ ]:
# Bron: Week 12 – Regression Time Series 3, opdracht 6 (Fourier-orde)
# Bron: Week 10 – Regression Time Series 2, opdracht 1 en week 10 –
#       application remarks (dummies, één categorie weglaten)
# Bron: Week 8 – slides Veelvoorkomende fouten (alleen numerieke
#       features standaardiseren)
# Prompt 7: Lineair model met dummies en log-target

WEATHER_NUM = ['temp', 'atemp', 'hum', 'windspeed', 'holiday']


def linear_model(categorical, numerical):
    """Lineaire regressie op log(1 + cnt) met dummies en getallen.

    Alleen numerieke kolommen worden gestandaardiseerd; categorieën worden
    dummies (one-hot, eerste categorie weggelaten).
    """
    prep = ColumnTransformer([
        ('dummies', OneHotEncoder(drop='first', handle_unknown='ignore'),
         categorical),
        ('numeriek', StandardScaler(), numerical)])
    pipeline = Pipeline([('prep', prep), ('model', LinearRegression())])
    return TransformedTargetRegressor(regressor=pipeline, func=np.log1p,
                                      inverse_func=np.expm1)


def fourier_columns(order, period='y'):
    """Namen van de Fourier-kolommen t/m een bepaalde orde."""
    return [f'{function}_{period}{k}' for k in range(1, order + 1)
            for function in ('sin', 'cos')]


order_rows = []
for order in [0, 1, 2, 3, 4, 6]:
    categorical = ['hour_wd', 'weekday', 'weathersit']
    numerical = ['trend'] + fourier_columns(order) + WEATHER_NUM
    score = cv_rmse(linear_model(categorical, numerical),
                    train_fe_part[categorical + numerical],
                    train_fe_part['cnt'])
    order_rows.append({'Fourier-orde jaar': order, 'CV-RMSE': score})
order_table = pd.DataFrame(order_rows)
best_row = order_table['CV-RMSE'].idxmin()
YEARLY_ORDER = int(order_table.loc[best_row, 'Fourier-orde jaar'])
display(order_table.round(1))
print('Beste orde voor het jaarseizoen:', YEARLY_ORDER)

numerical_base = ['trend'] + fourier_columns(YEARLY_ORDER) + WEATHER_NUM
day_options = {
    'uur-dummies': (['hour', 'weekday', 'weathersit'], ['workingday']),
    'dummies uur x werkdag': (['hour_wd', 'weekday', 'weathersit'], []),
    'Fourier-termen dag (orde 4)': (['weekday', 'weathersit'],
                                    ['workingday']
                                    + fourier_columns(4, period='d')),
}
day_rows = []
for name, (categorical, extra) in day_options.items():
    numerical = numerical_base + extra
    score = cv_rmse(linear_model(categorical, numerical),
                    train_fe_part[categorical + numerical],
                    train_fe_part['cnt'])
    day_rows.append({'dagpatroon': name, 'CV-RMSE': score})
pd.DataFrame(day_rows).round(1)

**Conclusie Fourier-orde en dagpatroon (lineaire regressie).**

- Een hogere orde voor het jaarseizoen helpt tot **orde 4** (CV-RMSE 63,6 → 59,3). Bij orde 6 wordt
  het weer slechter (60,0): de extra termen volgen toevallige schommelingen (overfitting). We kiezen
  **orde 4** (8 Fourier-kolommen).
- Het **dagpatroon** moet afhangen van het type dag. Met gewone uur-dummies is de fout 105,4, met
  **dummies voor uur × werkdag** 59,3. Fourier-termen voor de dag (orde 4) scoren het slechtst (113,2):
  ze zijn te glad voor de scherpe spitsen en hebben geen interactie met werkdag. Zoals opdracht 5 van
  *Regression Time Series 3* adviseert (De Haagse Hogeschool, z.d.-e), gebruiken we voor het
  dagpatroon óf dummies óf Fourier-termen; hier winnen de dummies.